# Decision State Review

Review every clustered decision-state dataset, inspect aligned train/test image sequences, and update `trajectory_criteria.csv` for the concrete `.npz` files behind the current decision state.

The criteria CSV is file-level, while a decision state is a timestep window. The helper functions below therefore edit the trajectory files used by the current decision-state view.

In [ ]:
from pathlib import Path
import gc

import numpy as np
import matplotlib.pyplot as plt
import torch
import ipywidgets as widgets
from IPython.display import clear_output, display

import trajectory_data
from nocode_robot_programming.state_decision.utils import Filename
from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import TrajectoryDatasetEvaluationViewBuilder
from nocode_robot_programming.state_decision_dataset_prepare.decision_state_clustering import cluster
from nocode_robot_programming.state_decision_dataset_prepare.trajectory_criteria import (
    TrajectoryCriteriaEditor,
    load_trajectory_criteria,
    sync_trajectory_criteria,
)

## Setup

In [ ]:
E = 10
ANOMALY = False
SCALE = 1.0
FPS = 20

STUDY_TASKS = ["peg_pick", "probe", "wrap"]
STUDY_MODALITIES = ["gst", "joy", "kin"]
EXCLUDED_USERS = {"additional"}
EXCLUDED_TASKS = {"test"}

EXPECTED_DEMO_VARIANTS = {"peg_pick": 2, "probe": 3, "wrap": 2}
EXPECTED_DECISION_STATES = {"peg_pick": 1, "probe": 2, "wrap": 1}

CRITERION_CHOICES = [
    ("Target image features not visible or <50 px", "target_features_not_visible_or_too_small"),
    ("Distractor", "distractor"),
    ("Very bad train data", "very_bad_train_data"),
    ("Camera not working", "camera_not_working"),
    ("Teaching multiple things at once", "teaching_multiple_things_at_once"),
    ("Hand in scene", "hand_in_scene"),
]

# None reviews only the study data: 8 users x 3 modalities x 3 tasks.
# You can also set a single task string or a list of tasks.
TASKS_TO_REVIEW = None
# TASKS_TO_REVIEW = "jan_joy_peg_pick"
# TASKS_TO_REVIEW = ["jan_joy_peg_pick", "jan_joy_probe"]

CRITERIA_CSV = Path(trajectory_data.package_path) / "trajectories" / "trajectory_criteria.csv"

dataset_builder = TrajectoryDatasetEvaluationViewBuilder()
criteria_editor = TrajectoryCriteriaEditor(dataset_builder, CRITERIA_CSV)

CRITERIA_CSV

## Study Summary

In [ ]:
def is_study_task_key(builder, task_key):
    user, modality, task_root = builder._split_user_modality_task(task_key)
    return (
        user not in EXCLUDED_USERS
        and task_root not in EXCLUDED_TASKS
        and task_root in STUDY_TASKS
        and modality in STUDY_MODALITIES
    )


def study_task_names(builder):
    return [task_key for task_key in builder.all_tasks if is_study_task_key(builder, task_key)]


def criteria_decisions():
    sync_trajectory_criteria(CRITERIA_CSV, dataset_builder.files)
    return load_trajectory_criteria(CRITERIA_CSV)


def decision_is_included(decisions, filename):
    decision = decisions.get(filename)
    return bool(decision and decision.use)


def criteria_parent_integrity_issues(tasks=None):
    decisions = criteria_decisions()
    tasks = list(dataset_builder.all_tasks if tasks is None else tasks)
    issues = []

    for task in tasks:
        rec = dataset_builder.tasks[task]
        demo_by_offset = {}
        demo_by_part = {}

        for name, offset, trial in zip(rec["names"], rec["offsets"], rec["trials"]):
            f = Filename(name)
            if int(trial) == -1:
                demo_by_offset[int(offset)] = f.filename
                demo_by_part[f.part_name] = f.filename

        for name in rec["names"]:
            f = Filename(name)
            child_file = f.filename
            if not decision_is_included(decisions, child_file):
                continue

            if f.trial >= 0:
                base_demo = demo_by_part.get(f.part_name)
                if base_demo and not decision_is_included(decisions, base_demo):
                    issues.append({
                        "task": task,
                        "child": child_file,
                        "required_parent": base_demo,
                        "reason": "included trial needs its base demo included",
                    })

            if f.offset != 0:
                parent_demo = demo_by_offset.get(f.parent_offset)
                if parent_demo is None:
                    issues.append({
                        "task": task,
                        "child": child_file,
                        "required_parent": None,
                        "reason": f"physical parent demo at offset {f.parent_offset} is missing",
                    })
                elif not decision_is_included(decisions, parent_demo):
                    issues.append({
                        "task": task,
                        "child": child_file,
                        "required_parent": parent_demo,
                        "reason": "included branch needs its parent demo included",
                    })

    return issues


def print_criteria_parent_integrity_issues(tasks=None, limit=80):
    issues = criteria_parent_integrity_issues(tasks=tasks)
    print(f"parent integrity issues: {len(issues)}")
    for issue in issues[:limit]:
        print(f"{issue['task']}: {issue['child']} -> {issue['required_parent']} ({issue['reason']})")
    if len(issues) > limit:
        print(f"... {len(issues) - limit} more")
    return issues


def repair_missing_parent_demos(tasks=None):
    issues = criteria_parent_integrity_issues(tasks=tasks)
    parents_to_include = sorted({
        issue["required_parent"]
        for issue in issues
        if issue["required_parent"] is not None
    })

    if not parents_to_include:
        print("No discarded parent demos need to be re-included.")
        return []

    included = criteria_editor.include(parents_to_include, print_report=True)
    print("Recreate filtered builders after this repair.")
    return included


def study_axes(*builders):
    task_keys = [task_key for builder in builders for task_key in study_task_names(builder)]
    users_found = {builders[0]._split_user_modality_task(task_key)[0] for task_key in task_keys}
    modalities_found = {builders[0]._split_user_modality_task(task_key)[1] for task_key in task_keys}
    tasks_found = {builders[0]._split_user_modality_task(task_key)[2] for task_key in task_keys}

    users = sorted(users_found)
    modalities = [modality for modality in STUDY_MODALITIES if modality in modalities_found]
    tasks = [task for task in STUDY_TASKS if task in tasks_found]
    return users, modalities, tasks


def summarize_builder(builder):
    summary = {}

    for task_key in study_task_names(builder):
        user, modality, task_root = builder._split_user_modality_task(task_key)
        rec = builder.tasks[task_key]

        demo_parts = sorted({
            Filename(name).part_name
            for name, trial in zip(rec["names"], rec["trials"])
            if int(trial) == -1
        })
        trial_rollouts = sum(1 for trial in rec["trials"] if int(trial) >= 0)

        summary[(user, modality, task_root)] = {
            "demo_variants": len(demo_parts),
            "trial_rollouts": int(trial_rollouts),
            "decision_states": len(cluster(rec, E)),
            "demo_parts": demo_parts,
        }

    return summary


def summary_matrices(raw_summary, filtered_summary, users, modalities, tasks, metric):
    columns = [(modality, task) for modality in modalities for task in tasks]
    raw = np.zeros((len(users), len(columns)), dtype=int)
    filtered = np.zeros_like(raw)

    for row, user in enumerate(users):
        for col, (modality, task) in enumerate(columns):
            key = (user, modality, task)
            raw[row, col] = raw_summary.get(key, {}).get(metric, 0)
            filtered[row, col] = filtered_summary.get(key, {}).get(metric, 0)

    return columns, raw, filtered


def print_study_summary_counts(raw_builder, filtered_builder):
    users, modalities, tasks = study_axes(raw_builder, filtered_builder)
    expected_tasks = len(users) * len(modalities) * len(tasks)
    expected_ds = len(users) * len(modalities) * sum(EXPECTED_DECISION_STATES[task] for task in tasks)

    raw_tasks = study_task_names(raw_builder)
    filtered_tasks = study_task_names(filtered_builder)
    raw_ds = sum(len(cluster(raw_builder.tasks[task], E)) for task in raw_tasks)
    filtered_ds = sum(len(cluster(filtered_builder.tasks[task], E)) for task in filtered_tasks)

    print(f"study users: {len(users)} {users}")
    print(f"study tasks raw: {len(raw_tasks)} / expected {expected_tasks}")
    print(f"study tasks after criteria: {len(filtered_tasks)} / expected {expected_tasks}")
    print(f"decision states raw: {raw_ds} / expected {expected_ds}")
    print(f"decision states after criteria: {filtered_ds} / expected {expected_ds}")


def plot_metric(ax, raw, filtered, users, columns, title, expected_by_task=None, vmax=None):
    if vmax is None:
        vmax = max(int(raw.max()), int(filtered.max()), 1)

    im = ax.imshow(filtered, cmap="YlGnBu", vmin=0, vmax=vmax)
    ax.set_title(f"{title} (raw -> after criteria)")
    ax.set_yticks(range(len(users)))
    ax.set_yticklabels(users)
    ax.set_xticks(range(len(columns)))
    ax.set_xticklabels([f"{modality}\n{task}" for modality, task in columns], rotation=0)
    ax.set_xticks(np.arange(-0.5, len(columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(users), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    for row in range(raw.shape[0]):
        for col, (_, task) in enumerate(columns):
            expected = None if expected_by_task is None else expected_by_task.get(task)
            mismatch = expected is not None and (raw[row, col] != expected or filtered[row, col] != expected)
            dropped = filtered[row, col] < raw[row, col]
            color = "crimson" if mismatch or dropped else ("white" if filtered[row, col] > vmax * 0.55 else "black")
            ax.text(col, row, f"{raw[row, col]}->{filtered[row, col]}", ha="center", va="center", color=color, fontsize=9)
            if mismatch:
                ax.add_patch(plt.Rectangle((col - 0.5, row - 0.5), 1, 1, fill=False, edgecolor="crimson", linewidth=2.0))

    return im


def plot_study_summary(raw_builder, filtered_builder):
    raw_summary = summarize_builder(raw_builder)
    filtered_summary = summarize_builder(filtered_builder)
    users, modalities, tasks = study_axes(raw_builder, filtered_builder)

    fig, axes = plt.subplots(3, 1, figsize=(1.35 * len(modalities) * len(tasks), 2.1 * len(users)), constrained_layout=True)

    metrics = [
        ("demo_variants", "Demonstration variants", EXPECTED_DEMO_VARIANTS, 3),
        ("decision_states", "Decision-state windows", EXPECTED_DECISION_STATES, 2),
        ("trial_rollouts", "Execution trials", None, None),
    ]

    for ax, (metric, title, expected, vmax) in zip(axes, metrics):
        columns, raw, filtered = summary_matrices(raw_summary, filtered_summary, users, modalities, tasks, metric)
        im = plot_metric(ax, raw, filtered, users, columns, title, expected_by_task=expected, vmax=vmax)
        fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)

    fig.suptitle("Study data only: excludes user='additional' and task='test'", fontsize=14)
    return fig

In [ ]:
parent_issues = print_criteria_parent_integrity_issues()

if parent_issues:
    print("\nRepair first, then rerun this cell:")
    print("repair_missing_parent_demos()")
else:
    dataset_builder_filtered_for_summary = TrajectoryDatasetEvaluationViewBuilder(
        criteria_csv=CRITERIA_CSV,
        sync_criteria_csv=True,
        print_criteria_report=False,
    )

    print_study_summary_counts(dataset_builder, dataset_builder_filtered_for_summary)
    plot_study_summary(dataset_builder, dataset_builder_filtered_for_summary);

In [ ]:
def review_tasks():
    if TASKS_TO_REVIEW is None:
        return study_task_names(dataset_builder)
    if isinstance(TASKS_TO_REVIEW, str):
        return [TASKS_TO_REVIEW]
    return list(TASKS_TO_REVIEW)


all_decision_states = [
    {"task": task, "ds_i": ds_i, **ds}
    for task in review_tasks()
    for ds_i, ds in enumerate(cluster(dataset_builder.tasks[task], E))
]

print(f"tasks: {len(review_tasks())}")
print(f"decision states: {len(all_decision_states)}")
all_decision_states[:3]

## Review Helpers

In [ ]:
review_i = 0
current_ds = None


def criteria_files(names):
    return [Filename(name).filename for name in names]


def files_for_decision_state(item):
    parts = set(item["relevant_parts"])
    return [
        name
        for name in dataset_builder.tasks[item["task"]]["names"]
        if Filename(name).part_name in parts
    ]


def split_files(files):
    train_files = []
    test_files = []

    for name in files:
        f = Filename(name)
        if ANOMALY:
            target = train_files if f.offset == 0 else test_files
        else:
            target = train_files if f.is_demo or f.trial == 0 else test_files
        target.append(name)

    return train_files, test_files


def load_decision_state(i):
    item = dict(all_decision_states[i])
    files = files_for_decision_state(item)
    train_files, test_files = split_files(files)
    relevant_parts = list(item["relevant_parts"])
    at = slice(item["start"], item["end"])

    d_train = dataset_builder.get_auto_dataset_view(
        file_names=train_files,
        relevant_parts=relevant_parts,
        at=at,
        anomaly=ANOMALY,
    )
    d_test = dataset_builder.get_auto_dataset_view(
        file_names=test_files,
        relevant_parts=relevant_parts,
        at=at,
        anomaly=ANOMALY,
    )

    if d_train is None and d_test is None:
        raise RuntimeError(f"No samples for decision state: {item}")
    if d_train is None:
        d_train = d_test
    if d_test is None:
        d_test = d_train

    f = Filename(relevant_parts[0])
    d_text = (
        f"{f.task_userstudy} {f.modality} {f.person}, "
        f"DS={item['ds_i']}, window={E}, anomaly={ANOMALY}, "
        f"train={len(train_files)}, test={len(test_files)}"
    )

    item.update(
        files=files,
        train_files=train_files,
        test_files=test_files,
        criteria_files=criteria_files(files),
        train_criteria_files=criteria_files(train_files),
        test_criteria_files=criteria_files(test_files),
        dataset=(d_train, d_test, d_text),
    )
    return item


def free_current_dataset():
    global current_ds
    current_ds = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def target_files(scope="test"):
    if current_ds is None:
        raise RuntimeError("Run show_current() first.")

    if scope == "test":
        return current_ds["test_criteria_files"]
    if scope == "task":
        return criteria_files(dataset_builder.tasks[current_ds["task"]]["names"])

    raise ValueError("scope must be one of: 'test', 'task'")


def status_current(scope="all"):
    return criteria_editor.status(target_files(scope), task=current_ds["task"])

## Cursor

In [ ]:
def show_current(i=None, scale=SCALE, fps=FPS):
    global review_i, current_ds

    if i is not None:
        review_i = max(0, min(int(i), len(all_decision_states) - 1))

    free_current_dataset()
    current_ds = load_decision_state(review_i)
    d_train, d_test, d_text = current_ds["dataset"]

    clear_output(wait=True)
    print(f"[{review_i + 1}/{len(all_decision_states)}] {current_ds['task']}")
    print(f"window={current_ds['start']}:{current_ds['end']}")
    print(f"parts={current_ds['relevant_parts']}")
    print(d_text)
    print()
    print(
        "edit scope counts: "
        f"test={len(current_ds['test_criteria_files'])}, "
        f"train={len(current_ds['train_criteria_files'])}, "
        f"all={len(current_ds['criteria_files'])}"
    )


    print("\nTRAIN")
    d_train.showcase_aligned(fps=fps, scale=scale)

    print("\nTEST")
    d_test.showcase_aligned(fps=fps, scale=scale)

    print("\ncriteria status:")
    for row in status_current("all"):
        print(row)

    return current_ds


def next_current(scale=SCALE, fps=FPS):
    return show_current(review_i + 1, scale=scale, fps=fps)


def prev_current(scale=SCALE, fps=FPS):
    return show_current(review_i - 1, scale=scale, fps=fps)


def go(i, scale=SCALE, fps=FPS):
    return show_current(i, scale=scale, fps=fps)

## Actions

In [ ]:
def discard_current(reason="", criterion=None, scope="test"):
    if criterion is None:
        criterion = CRITERION_CHOICES[0][1]

    edited = criteria_editor.discard(
        target_files(scope),
        reason=reason,
        criterion=criterion,
        task=current_ds["task"],
    )
    print(f"discarded {len(edited)} file(s) from scope={scope}, criterion={criterion}")
    return edited


def include_current(scope="test"):
    edited = criteria_editor.include(
        target_files(scope),
        task=current_ds["task"],
    )
    print(f"included {len(edited)} file(s) from scope={scope}")
    return edited


def discard_test(reason="", criterion=None):
    return discard_current(reason=reason, criterion=criterion, scope="test")


def include_test():
    return include_current(scope="test")

## Widget Review Loop

In [ ]:
review_output = widgets.Output()
action_output = widgets.Output()

index_widget = widgets.BoundedIntText(
    value=0,
    min=0,
    max=max(len(all_decision_states) - 1, 0),
    description="Index",
    layout=widgets.Layout(width="170px"),
)
scale_widget = widgets.FloatText(
    value=SCALE,
    description="Scale",
    layout=widgets.Layout(width="150px"),
)
fps_widget = widgets.IntText(
    value=FPS,
    description="FPS",
    layout=widgets.Layout(width="130px"),
)
scope_widget = widgets.Dropdown(
    options=[("current branch/test files", "test"), ("whole task", "task")],
    value="test",
    description="Scope",
    layout=widgets.Layout(width="260px"),
)
criterion_widget = widgets.Dropdown(
    options=CRITERION_CHOICES,
    value=CRITERION_CHOICES[0][1],
    description="Criterion",
    layout=widgets.Layout(width="560px"),
)
reason_widget = widgets.Textarea(
    value="",
    placeholder="Optional short note; the selected criterion is written separately.",
    description="Note",
    layout=widgets.Layout(width="560px", height="72px"),
)

show_button = widgets.Button(description="Show", icon="eye", button_style="info")
prev_button = widgets.Button(description="Previous", icon="arrow-left")
next_button = widgets.Button(description="Next", icon="arrow-right")
include_button = widgets.Button(description="Include", icon="check", button_style="success")
discard_button = widgets.Button(description="Discard", icon="trash", button_style="danger")


def render_current(i=None):
    if not all_decision_states:
        with review_output:
            clear_output(wait=True)
            print("No decision states to review. Check TASKS_TO_REVIEW and study filters.")
        return None

    with review_output:
        clear_output(wait=True)
        item = show_current(i if i is not None else index_widget.value, scale=scale_widget.value, fps=fps_widget.value)

    if index_widget.value != review_i:
        index_widget.value = review_i
    return item


def ensure_current_loaded():
    if current_ds is None:
        render_current(index_widget.value)


def run_action(action_name):
    ensure_current_loaded()
    scope = scope_widget.value
    criterion = criterion_widget.value
    reason = reason_widget.value.strip()

    with action_output:
        clear_output(wait=True)
        if action_name == "include":
            edited = include_current(scope=scope)
        elif action_name == "discard":
            edited = discard_current(reason=reason, criterion=criterion, scope=scope)
        else:
            raise ValueError(action_name)
        print("\nEdited files:")
        for filename in edited:
            print(f"  {filename}")


def on_show_clicked(_):
    render_current(index_widget.value)


def on_prev_clicked(_):
    render_current(review_i - 1)


def on_next_clicked(_):
    render_current(review_i + 1)


def on_include_clicked(_):
    run_action("include")


def on_discard_clicked(_):
    run_action("discard")


show_button.on_click(on_show_clicked)
prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)
include_button.on_click(on_include_clicked)
discard_button.on_click(on_discard_clicked)

review_widget = widgets.VBox([
    widgets.HTML("<b>Decision-state review</b>"),
    widgets.HBox([prev_button, next_button, index_widget, show_button]),
    widgets.HBox([scale_widget, fps_widget, scope_widget]),
    criterion_widget,
    reason_widget,
    widgets.HBox([include_button, discard_button]),
    action_output,
    review_output,
])

display(review_widget)
_ = render_current(0)

Use **Scope** to choose whether the action edits only the current branch/test files or the whole task. The widget no longer exposes train/root-only or current-DS-all edits because those can leave an included branch without its parent demo. **Discard** writes the selected **Criterion** into the CSV `criterion` column. **Note** is optional and is written into `discard_reason`. Actions do not move or refresh the current decision state; use **Next**, **Previous**, or **Show** explicitly.

## Criteria Integrity Repair

In [ ]:
parent_issues = print_criteria_parent_integrity_issues()
# If this prints issues caused by accidentally discarded parent demos, run:
# repair_missing_parent_demos()

## Apply Criteria Filter

In [ ]:
# Recreate the builder after editing the CSV so use=0 rows are filtered out.
dataset_builder_filtered = TrajectoryDatasetEvaluationViewBuilder(
    criteria_csv=CRITERIA_CSV,
    sync_criteria_csv=True,
    print_criteria_report=True,
)

print(f"unfiltered trajectories: {len(dataset_builder.names)}")
print(f"filtered trajectories: {len(dataset_builder_filtered.names)}")